In [ ]:
import subprocess, sys

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "libxcb1", "libxcb-shm0", "libxcb-render0",
                 "libxcb-xfixes0", "libgl1-mesa-glx", "libglib2.0-0", "libsm6", "libxext6", "libxrender1"],
               check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "ultralytics", "opencv-python-headless", "numpy"], check=True)

print("All dependencies installed ✓")

In [ ]:
import json
import os
import time
from pathlib import Path

import torch
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = getattr(gpu, "total_memory", getattr(gpu, "total_mem", 0)) / 1024**3
    print(f"GPU: {gpu.name}  |  VRAM: {vram_gb:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: No CUDA GPU detected — inference will be slow on CPU")

MODEL_PATH = "best.pt"
model = YOLO(MODEL_PATH)
model.to(DEVICE)
print(f"Model loaded on {DEVICE}  |  Classes: {model.names}")

In [ ]:
BASE_DIR = Path(".")

VIDEOS = [
    {
        "name": "Video 1",
        "video": BASE_DIR / "1Corrected/corrected/1.mp4",
        "tracks_json": BASE_DIR / "1Corrected/corrected/corrected_tracked_tracks.json",
        "output_json": BASE_DIR / "1Corrected/corrected/combined_tracks_and_moves.json",
    },
    {
        "name": "Video 2",
        "video": BASE_DIR / "2Corrected/corrected/2.mp4",
        "tracks_json": BASE_DIR / "2Corrected/corrected/corrected_tracked_tracks.json",
        "output_json": BASE_DIR / "2Corrected/corrected/combined_tracks_and_moves.json",
    },
    {
        "name": "Video 3",
        "video": BASE_DIR / "3Corrected/corrected/3.mp4",
        "tracks_json": BASE_DIR / "3Corrected/corrected/corrected_tracked_tracks.json",
        "output_json": BASE_DIR / "3Corrected/corrected/combined_tracks_and_moves.json",
    },
    {
        "name": "Video 4",
        "video": BASE_DIR / "4Corrected/corrected/4.mp4",
        "tracks_json": BASE_DIR / "4Corrected/corrected/corrected_tracked_tracks.json",
        "output_json": BASE_DIR / "4Corrected/corrected/combined_tracks_and_moves.json",
    },
    {
        "name": "Video 5",
        "video": BASE_DIR / "5Corrected/corrected/5.mp4",
        "tracks_json": BASE_DIR / "5Corrected/corrected/corrected_tracked_tracks.json",
        "output_json": BASE_DIR / "5Corrected/corrected/combined_tracks_and_moves.json",
    },
]

for v in VIDEOS:
    assert v["video"].exists(), f"Missing video: {v['video']}"
    assert v["tracks_json"].exists(), f"Missing JSON: {v['tracks_json']}"

print(f"All {len(VIDEOS)} video/json pairs found ✓")

In [ ]:
import cv2
import numpy as np
from PIL import Image

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.10
BATCH_SIZE = 32

ROTATE_DEGREES = -0.55
NET_FOOT_Y = 547
KEEP_COURT_SIDE = "near"

PIL_BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC


def build_rotation_params(width, height, angle_deg):
    pil_angle = -angle_deg
    mask = Image.new("RGBA", (width, height), (255, 255, 255, 255))
    rotated_mask = mask.rotate(pil_angle, resample=PIL_BICUBIC, expand=True, fillcolor=(0, 0, 0, 0))
    bound_w, bound_h = rotated_mask.size
    alpha = np.array(rotated_mask.getchannel("A"))
    valid = (alpha > 0).astype(np.uint8)
    integral = cv2.integral(valid)
    target_aspect = width / height

    def region_all_valid(x1, y1, crop_w, crop_h):
        x2 = x1 + crop_w
        y2 = y1 + crop_h
        total = integral[y2, x2] - integral[y1, x2] - integral[y2, x1] + integral[y1, x1]
        return total == crop_w * crop_h

    lo, hi = 1, bound_h
    best_w, best_h = width, height
    while lo <= hi:
        crop_h = (lo + hi) // 2
        crop_w = int(round(crop_h * target_aspect))
        if crop_w > bound_w:
            hi = crop_h - 1
            continue
        x1 = (bound_w - crop_w) // 2
        y1 = (bound_h - crop_h) // 2
        if region_all_valid(x1, y1, crop_w, crop_h):
            best_w, best_h = crop_w, crop_h
            lo = crop_h + 1
        else:
            hi = crop_h - 1

    return {
        "bound_w": bound_w, "bound_h": bound_h,
        "crop_x1": (bound_w - best_w) // 2, "crop_y1": (bound_h - best_h) // 2,
        "crop_w": best_w, "crop_h": best_h,
    }


def preprocess_frame(frame_bgr, params, output_size, angle_deg):
    if abs(float(angle_deg)) < 1e-6:
        return frame_bgr
    pil_angle = -float(angle_deg)
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    rotated = pil_image.rotate(pil_angle, resample=PIL_BICUBIC, expand=True, fillcolor=(0, 0, 0))
    x1, y1 = params["crop_x1"], params["crop_y1"]
    cropped = rotated.crop((x1, y1, x1 + params["crop_w"], y1 + params["crop_h"]))
    if cropped.size != output_size:
        cropped = cropped.resize(output_size, resample=PIL_BICUBIC)
    return cv2.cvtColor(np.array(cropped), cv2.COLOR_RGB2BGR)


def mask_far_side(frame_bgr, net_y):
    """Black out everything above the net line so best.pt can't see the far court."""
    masked = frame_bgr.copy()
    masked[:net_y, :] = 0
    return masked


def run_inference(video_path: Path) -> dict[int, list[dict]]:
    """
    Run best.pt on a video with the same rotation/crop as the tracker,
    plus masking out the far side of the court. Returns {frame_index: [detections]}.
    """
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open: {video_path}"
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    rot_params = build_rotation_params(orig_w, orig_h, ROTATE_DEGREES)
    output_size = (orig_w, orig_h)

    move_detections: dict[int, list[dict]] = {}
    frame_idx = 0
    batch_frames = []
    batch_indices = []

    while True:
        ret, raw_frame = cap.read()
        if not ret:
            break

        processed = preprocess_frame(raw_frame, rot_params, output_size, ROTATE_DEGREES)
        masked = mask_far_side(processed, NET_FOOT_Y)

        batch_frames.append(masked)
        batch_indices.append(frame_idx)
        frame_idx += 1

        if len(batch_frames) >= BATCH_SIZE or not ret:
            results = model.predict(
                source=batch_frames,
                device=DEVICE,
                conf=CONF_THRESHOLD,
                imgsz=1280,
                verbose=False,
                half=DEVICE == "cuda",
            )
            for bi, r in enumerate(results):
                boxes = r.boxes
                if boxes is None or len(boxes) == 0:
                    continue
                frame_dets = []
                for i in range(len(boxes)):
                    xyxy = boxes.xyxy[i].tolist()
                    cls_id = int(boxes.cls[i])
                    conf = float(boxes.conf[i])
                    frame_dets.append({
                        "bbox_xyxy": [int(round(c)) for c in xyxy],
                        "class": model.names[cls_id],
                        "conf": round(conf, 4),
                    })
                move_detections[batch_indices[bi]] = frame_dets
            batch_frames.clear()
            batch_indices.clear()

        if frame_idx % 500 == 0:
            print(f"    {frame_idx}/{total} frames processed...")

    if batch_frames:
        results = model.predict(
            source=batch_frames,
            device=DEVICE,
            conf=CONF_THRESHOLD,
            imgsz=1280,
            verbose=False,
            half=DEVICE == "cuda",
        )
        for bi, r in enumerate(results):
            boxes = r.boxes
            if boxes is None or len(boxes) == 0:
                continue
            frame_dets = []
            for i in range(len(boxes)):
                xyxy = boxes.xyxy[i].tolist()
                cls_id = int(boxes.cls[i])
                conf = float(boxes.conf[i])
                frame_dets.append({
                    "bbox_xyxy": [int(round(c)) for c in xyxy],
                    "class": model.names[cls_id],
                    "conf": round(conf, 4),
                })
            move_detections[batch_indices[bi]] = frame_dets

    cap.release()
    return move_detections


def compute_iou(box_a: list, box_b: list) -> float:
    """IoU between two [x1, y1, x2, y2] boxes."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter == 0:
        return 0.0
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    return inter / (area_a + area_b - inter)


def compute_containment(move_box: list, person_box: list) -> float:
    """
    Fraction of the move box that falls inside the person box.
    Useful when the move detection is small and fully inside a larger person box
    (IoU would be low, but containment would be high).
    """
    x1 = max(move_box[0], person_box[0])
    y1 = max(move_box[1], person_box[1])
    x2 = min(move_box[2], person_box[2])
    y2 = min(move_box[3], person_box[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    move_area = max(1, (move_box[2] - move_box[0]) * (move_box[3] - move_box[1]))
    return inter / move_area


def map_moves_to_ids(
    tracks: list[dict],
    move_detections: dict[int, list[dict]],
) -> list[dict]:
    """
    Merge move detections into the existing tracks JSON.
    Each person detection gets a "moves" list (possibly empty).
    Multiple people can receive the same move class on the same frame.
    """
    frame_lookup = {entry["frame"]: entry for entry in tracks}

    for entry in tracks:
        for det in entry["detections"]:
            det["moves"] = []

    for frame_idx, moves in move_detections.items():
        frame_data = frame_lookup.get(frame_idx)
        if frame_data is None:
            continue

        person_dets = frame_data["detections"]
        if not person_dets:
            continue

        for move in moves:
            best_score = 0.0
            best_person = None

            for person in person_dets:
                iou = compute_iou(move["bbox_xyxy"], person["bbox_xyxy"])
                containment = compute_containment(move["bbox_xyxy"], person["bbox_xyxy"])
                score = max(iou, containment)

                if score > best_score:
                    best_score = score
                    best_person = person

            if best_person is not None and best_score >= IOU_THRESHOLD:
                best_person["moves"].append({
                    "action": move["class"],
                    "confidence": move["conf"],
                    "move_bbox_xyxy": move["bbox_xyxy"],
                    "overlap_score": round(best_score, 4),
                })

    return tracks


def process_video(video_cfg: dict) -> str:
    """Full pipeline for one video: infer → map → save."""
    name = video_cfg["name"]
    print(f"\n{'='*60}")
    print(f"  Processing: {name}")
    print(f"{'='*60}")

    t0 = time.time()
    with open(video_cfg["tracks_json"]) as f:
        tracks = json.load(f)
    n_frames = len(tracks)
    print(f"  Loaded {n_frames} frames of tracking data")

    t1 = time.time()
    print(f"  Running best.pt inference (batch={BATCH_SIZE}, device={DEVICE})...")
    move_dets = run_inference(video_cfg["video"])
    t2 = time.time()
    total_moves = sum(len(v) for v in move_dets.values())
    frames_with_moves = len(move_dets)
    print(f"  Inference done: {total_moves} move detections across {frames_with_moves} frames")
    print(f"  Inference time: {t2 - t1:.1f}s ({n_frames / max(0.1, t2 - t1):.1f} FPS)")

    combined = map_moves_to_ids(tracks, move_dets)
    t3 = time.time()

    assigned = sum(
        1 for entry in combined
        for det in entry["detections"]
        if det["moves"]
    )
    print(f"  Mapped moves to {assigned} person-frame instances")

    out_path = video_cfg["output_json"]
    with open(out_path, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"  Saved: {out_path}")
    print(f"  Total time: {t3 - t0:.1f}s")

    return str(out_path)


print("Functions defined ✓")

In [ ]:
output_files = []
total_start = time.time()

for vcfg in VIDEOS:
    out = process_video(vcfg)
    output_files.append(out)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"  ALL DONE — {len(output_files)} videos processed in {total_elapsed:.1f}s")
print(f"{'='*60}")
for p in output_files:
    print(f"  → {p}")

In [ ]:
with open(output_files[0]) as f:
    combined = json.load(f)

frames_with_moves = [
    entry for entry in combined
    if any(det["moves"] for det in entry["detections"])
]

if frames_with_moves:
    sample = frames_with_moves[0]
    print(json.dumps(sample, indent=2))
    print(f"\n... {len(frames_with_moves)} total frames have at least one move detected")
else:
    print("No moves were detected — try lowering CONF_THRESHOLD")

In [ ]:
from collections import Counter

for vcfg in VIDEOS:
    with open(vcfg["output_json"]) as f:
        data = json.load(f)

    action_counts = Counter()
    unassigned_moves = 0
    multi_move_persons = 0

    for entry in data:
        for det in entry["detections"]:
            n = len(det["moves"])
            if n > 0:
                multi_move_persons += 1 if n > 1 else 0
                for m in det["moves"]:
                    action_counts[m["action"]] += 1

    print(f"\n{vcfg['name']}:")
    print(f"  Total frames: {len(data)}")
    for action in sorted(action_counts):
        print(f"  {action:>8s}: {action_counts[action]}")
    total = sum(action_counts.values())
    print(f"  {'TOTAL':>8s}: {total}")
    if multi_move_persons:
        print(f"  ({multi_move_persons} person-frames had multiple moves assigned)")

In [ ]:
def id_to_color(track_id: int) -> tuple:
    """Deterministic distinct color per track ID."""
    rng = np.random.RandomState(track_id * 31 + 7)
    return tuple(int(c) for c in rng.randint(80, 255, size=3))


ACTION_COLORS = {
    "block":  (0, 165, 255),
    "serve":  (0, 255, 0),
    "set":    (255, 255, 0),
    "attack": (0, 0, 255),
    "dig":    (255, 0, 255),
}


def render_overlay_video(video_cfg: dict):
    """Read original video + combined JSON, apply same rotation/crop, draw boxes/labels, write new video."""
    name = video_cfg["name"]
    video_path = str(video_cfg["video"])
    json_path = str(video_cfg["output_json"])
    out_dir = video_cfg["output_json"].parent
    out_path = str(out_dir / "overlay_with_moves.mp4")

    print(f"\n  Rendering overlay: {name}")

    with open(json_path) as f:
        combined = json.load(f)
    frame_lookup = {entry["frame"]: entry for entry in combined}

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    rot_params = build_rotation_params(w, h, ROTATE_DEGREES)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

    font = cv2.FONT_HERSHEY_SIMPLEX
    thickness_box = 2
    font_scale_id = 0.7
    font_scale_move = 0.6

    frame_idx = 0
    while True:
        ret, raw_frame = cap.read()
        if not ret:
            break

        frame = preprocess_frame(raw_frame, rot_params, (w, h), ROTATE_DEGREES)

        entry = frame_lookup.get(frame_idx)
        if entry is not None:
            for det in entry["detections"]:
                x1, y1, x2, y2 = det["bbox_xyxy"]
                tid = det["id"]
                color = id_to_color(tid)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness_box)

                label = f"ID {tid}"
                (lw, lh), _ = cv2.getTextSize(label, font, font_scale_id, 2)
                cv2.rectangle(frame, (x1, y1 - lh - 8), (x1 + lw + 4, y1), color, -1)
                cv2.putText(frame, label, (x1 + 2, y1 - 4), font, font_scale_id,
                            (0, 0, 0), 2, cv2.LINE_AA)

                if det.get("moves"):
                    y_offset = y2 + 20
                    for move in det["moves"]:
                        action = move["action"]
                        conf = move["confidence"]
                        move_label = f"{action} ({conf:.0%})"
                        action_color = ACTION_COLORS.get(action, color)

                        (mw, mh), _ = cv2.getTextSize(move_label, font, font_scale_move, 2)
                        cv2.rectangle(frame, (x1, y_offset - mh - 4),
                                      (x1 + mw + 4, y_offset + 4), action_color, -1)
                        cv2.putText(frame, move_label, (x1 + 2, y_offset), font,
                                    font_scale_move, (0, 0, 0), 2, cv2.LINE_AA)
                        y_offset += mh + 12

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 500 == 0:
            print(f"    {frame_idx}/{total_frames} frames rendered...")

    cap.release()
    writer.release()
    print(f"  Saved: {out_path} ({frame_idx} frames)")
    return out_path


print("Overlay renderer defined ✓")

In [ ]:
print("Rendering overlay videos for all clips...")
overlay_start = time.time()

overlay_files = []
for vcfg in VIDEOS:
    out = render_overlay_video(vcfg)
    overlay_files.append(out)

overlay_elapsed = time.time() - overlay_start
print(f"\n{'='*60}")
print(f"  ALL OVERLAYS DONE in {overlay_elapsed:.1f}s")
print(f"{'='*60}")
for p in overlay_files:
    print(f"  → {p}")